# PQC IDS: reproduce the three seed-0 C1-SF fits

Your training-readiness check passed. This notebook takes the next step: fit
Random Forest, XGBoost and MLP for **split seed 0 only**, using the recorded
stable-feature subset and the unchanged model constructors.

1. Open this notebook in Google Colab and select **Runtime → Run all**.
2. Upload the updated **PQC_IDS_Reproduction_Working_Copy.zip** when prompted.
3. Connect the Google Drive account containing the four CSV files. The default
   data folder is `/content/drive/MyDrive/cicids2018`.
4. Return the downloaded **PQC_IDS_Seed0_Results.zip**, including if a check fails.

The notebook installs a separate Python 3.12.13 runtime and the recorded
NumPy, pandas, SciPy, scikit-learn and XGBoost versions. Your Colab notebook
kernel keeps its own environment. Internet access and several GB of free local
disk are needed for the packages and temporary CSV copies; fitting uses CPU.
No duration estimate is promised.

Results and logs are written to a new folder under
`MyDrive/pqc_ids_reproduction_checks/`. The original CSVs and historical results
are only read. This run does not refit feature selection, reproduce all ten
splits, or establish that the full paper is ready for submission.

In [ ]:
from pathlib import Path, PurePosixPath
from datetime import datetime, timezone
import hashlib
import json
import os
import platform
import shutil
import stat
import subprocess
import sys
import tempfile
import traceback
import uuid
import zipfile
from google.colab import files, drive

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:8]
SESSION = Path(tempfile.mkdtemp(prefix="pqc_seed0_", dir="/content"))

def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

uploaded = files.upload()
zip_names = [name for name in uploaded if name.lower().endswith(".zip")]
if len(zip_names) != 1:
    raise ValueError("Upload exactly one updated working-copy ZIP.")
archive_path = SESSION / "working_copy.zip"
archive_path.write_bytes(uploaded[zip_names[0]])
del uploaded
extracted = SESSION / "package"
with zipfile.ZipFile(archive_path) as archive:
    if archive.testzip() is not None:
        raise ValueError("The ZIP failed its integrity check. Download the working copy again.")
    for info in archive.infolist():
        path = PurePosixPath(info.filename)
        if path.is_absolute() or ".." in path.parts or stat.S_ISLNK(info.external_attr >> 16):
            raise ValueError("Unexpected archive path: " + info.filename)
    archive.extractall(extracted)
manifest_paths = list(extracted.glob("*/package_manifest.json"))
if len(manifest_paths) != 1:
    raise ValueError("Expected one working-copy package manifest.")
PACKAGE = manifest_paths[0].parent
manifest = json.loads(manifest_paths[0].read_text())
listed = set()
for item in manifest["files"]:
    path = (PACKAGE / item["path"]).resolve()
    if not path.is_relative_to(PACKAGE.resolve()):
        raise ValueError("Invalid manifest path.")
    if not path.is_file() or path.stat().st_size != item["bytes"] or sha256(path) != item["sha256"]:
        raise ValueError("Package file does not match the manifest: " + item["path"])
    listed.add(item["path"])
actual = {p.relative_to(PACKAGE).as_posix() for p in PACKAGE.rglob("*") if p.is_file()}
if actual != listed | {"package_manifest.json"} or len(listed) != len(manifest["files"]):
    raise ValueError("The archive contains unlisted files or duplicate manifest records.")
if not (PACKAGE / "scripts/seed0_reproduction.py").is_file():
    raise ValueError("This is an older working copy. Use the ZIP provided with this notebook.")
print("Working-copy files verified:", len(listed))
print("Run identifier:", RUN_ID)

## Connect your data and create a fresh result folder

If your CSV folder differs, edit `DATA_DIR` before running the next cell. Leave
the four CSV basenames unchanged. Each run creates a separate result folder.

In [ ]:
drive.mount("/content/drive")
DATA_DIR = Path("/content/drive/MyDrive/cicids2018")
DRIVE_OUTPUT = Path("/content/drive/MyDrive/pqc_ids_reproduction_checks") / RUN_ID
DRIVE_OUTPUT.mkdir(parents=True, exist_ok=False)
RUN_OUTPUT = DRIVE_OUTPUT / "results"
TARGET = json.loads((PACKAGE / "environment/seed0_known_versions.json").read_text())
SUMMARY = {
    "run_id": RUN_ID,
    "status": "preparing_environment",
    "colab_kernel_python": platform.python_version(),
    "package_sha256": sha256(archive_path),
    "package_manifest_sha256": sha256(PACKAGE / "package_manifest.json"),
    "data_directory": str(DATA_DIR),
    "result_directory": str(DRIVE_OUTPUT),
    "scope": "Three seed-0 C1-SF fits only; no full-experiment reproduction claim",
    "target": TARGET,
}

def save_summary():
    (DRIVE_OUTPUT / "notebook_status.json").write_text(json.dumps(SUMMARY, indent=2) + "\n")

save_summary()
shutil.copy2(PACKAGE / "package_manifest.json", DRIVE_OUTPUT / "working_copy_manifest.json")
shutil.copy2(PACKAGE / "environment/seed0_known_versions.json", DRIVE_OUTPUT / "seed0_known_versions.json")
shutil.copy2(PACKAGE / "environment/seed0_requirements.txt", DRIVE_OUTPUT / "seed0_requirements.txt")
print("Results will be saved to:", DRIVE_OUTPUT)

## Install the recorded top-level versions

This restores the recorded Python and package version numbers. It does **not**
reconstruct the historical OS, CPU, BLAS build or complete dependency lock.
The runner records those details for the new attempt. If installation fails,
training is skipped and the last cell still exports the log.

Python: **3.12.13** · NumPy: **2.0.2** · pandas: **2.2.2** · SciPy: **1.16.3** ·
scikit-learn: **1.6.1** · XGBoost: **3.2.0**.
SciPy's version comes from the saved statistical configuration; the other
versions come from the C1-SF fitting-environment record. Statsmodels is not
needed because this check uses the recorded features rather than rerunning KS
or mutual-information selection.

The setup uses uv's documented [Python installation](https://docs.astral.sh/uv/guides/install-python/)
and [isolated environment](https://docs.astral.sh/uv/pip/environments/) commands.

In [ ]:
RUN_ENV = os.environ.copy()
for name in ("PYTHONPATH", "PYTHONHOME", "VIRTUAL_ENV"):
    RUN_ENV.pop(name, None)
RUN_ENV["PYTHONNOUSERSITE"] = "1"
RUN_ENV["UV_PYTHON_INSTALL_DIR"] = str(SESSION / "python_runtimes")
RUN_ENV["UV_CACHE_DIR"] = str(SESSION / "uv_cache")

def run_logged(command, log_name, env):
    with (DRIVE_OUTPUT / log_name).open("w", encoding="utf-8") as log:
        log.write("Command: " + json.dumps([str(s) for s in command]) + "\n")
        log.flush()
        with subprocess.Popen([str(s) for s in command], stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, stdin=subprocess.DEVNULL,
                              text=True, bufsize=1, env=env) as process:
            for line in process.stdout:
                print(line, end="", flush=True)
                log.write(line)
                log.flush()
            return process.wait()

ENVIRONMENT_READY = False
VENV = SESSION / "seed0_environment"
FIT_PYTHON = VENV / "bin/python"
try:
    tooling = SESSION / "uv_tooling"
    bootstrap = [sys.executable, "-m", "pip", "install", "--target", tooling,
                 "--no-deps", "uv==0.12.15"]
    if run_logged(bootstrap, "01_uv_install.log", RUN_ENV) != 0:
        raise RuntimeError("Could not install the pinned uv tool.")
    uv_env = RUN_ENV.copy()
    uv_env["PYTHONPATH"] = str(tooling)
    uv = [sys.executable, "-m", "uv"]
    create = uv + ["venv", "--python", TARGET["python"], "--managed-python", VENV]
    if run_logged(create, "02_python_setup.log", uv_env) != 0:
        raise RuntimeError("Could not create the requested Python runtime.")
    install = uv + ["pip", "install", "--python", FIT_PYTHON, "--only-binary", ":all:",
                    "-r", PACKAGE / "environment/seed0_requirements.txt"]
    if run_logged(install, "03_packages_install.log", uv_env) != 0:
        raise RuntimeError("Could not install the recorded package versions.")
    ENVIRONMENT_READY = True
    SUMMARY["status"] = "environment_installed"
except Exception as error:
    SUMMARY["status"] = "environment_setup_failed"
    SUMMARY["error"] = str(error)
    (DRIVE_OUTPUT / "setup_traceback.txt").write_text(traceback.format_exc())
    print("Setup failed:", error)
finally:
    save_summary()
print("Environment ready:", ENVIRONMENT_READY)

## Fit and compare the three seed-0 models

This cell **trains three models**. It first checks the pinned versions, original
source hashes, data counts, split indices and selected-feature order. Input
CSV hashes and a prepared-matrix hash are recorded as identities for this
attempt; no historical matrix hash is available for an exact data comparison.

The comparison rule is fixed before this run:

| Check | Required agreement |
|---|---|
| Weighted F1 and each class F1 | Absolute difference ≤ 1e-12 |
| Prediction bytes | SHA-256 must match the recorded model's predictions |
| Weighted multiclass ROC-AUC | Absolute difference ≤ 5e-5; exact difference retained |
| Probability bytes | Hash comparison recorded separately; not required to be identical |

The ROC-AUC threshold is a new reproduction-screening tolerance, not a margin
for the paper's equivalence tests. A passing result means only that these
three fits met these checks. Any mismatch is retained for review; this
notebook never proceeds automatically to a full experiment.

In [ ]:
if ENVIRONMENT_READY:
    try:
        SUMMARY["status"] = "running_seed0"
        save_summary()
        command = [FIT_PYTHON, "-u", PACKAGE / "scripts/seed0_reproduction.py",
                   "--data-dir", DATA_DIR, "--output", RUN_OUTPUT,
                   "--cache-dir", SESSION / "input_cache"]
        returncode = run_logged(command, "04_seed0_run.log", RUN_ENV)
        SUMMARY["seed0_returncode"] = returncode
        status_path = RUN_OUTPUT / "seed0_status.json"
        if status_path.is_file():
            result_status = json.loads(status_path.read_text())
            SUMMARY["status"] = result_status["status"]
            SUMMARY["fits_completed"] = result_status["fits_completed"]
            print("\nSeed-0 result:", SUMMARY["status"])
        else:
            SUMMARY["status"] = "runner_failed_before_status"
        comparison = RUN_OUTPUT / "seed0_comparison.csv"
        if comparison.is_file():
            print(comparison.read_text())
    except Exception as error:
        SUMMARY["status"] = "runner_exception"
        SUMMARY["error"] = str(error)
        (DRIVE_OUTPUT / "runner_traceback.txt").write_text(traceback.format_exc())
        print("Runner failed:", error)
    finally:
        save_summary()
else:
    print("Training skipped because the requested environment was not installed.")

## Export the result and logs

Run this cell even after a failure. The ZIP includes status records, versions,
input hashes, comparison scores, predictions and warnings where available.
The CSV inputs, downloaded runtimes and installation caches are excluded.

If the browser download does not start, the same ZIP is saved in the Drive
result folder printed below. Upload **PQC_IDS_Seed0_Results.zip** for review.

In [ ]:
SUMMARY["exported_utc"] = datetime.now(timezone.utc).isoformat()
save_summary()
result_files = sorted(p for p in DRIVE_OUTPUT.rglob("*")
                      if p.is_file() and p.name not in ("PQC_IDS_Seed0_Results.zip", "result_manifest.json"))
manifest = [{"path": p.relative_to(DRIVE_OUTPUT).as_posix(), "bytes": p.stat().st_size,
             "sha256": sha256(p)} for p in result_files]
manifest_file = DRIVE_OUTPUT / "result_manifest.json"
manifest_file.write_text(json.dumps({"run_id": RUN_ID, "files": manifest}, indent=2) + "\n")
result_files.append(manifest_file)
result_zip = SESSION / "PQC_IDS_Seed0_Results.zip"
with zipfile.ZipFile(result_zip, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in result_files:
        archive.write(path, path.relative_to(DRIVE_OUTPUT).as_posix())
with zipfile.ZipFile(result_zip) as archive:
    if archive.testzip() is not None:
        raise RuntimeError("The result archive failed its integrity check.")
saved_zip = DRIVE_OUTPUT / result_zip.name
shutil.copy2(result_zip, saved_zip)
print("Final status:", SUMMARY["status"])
print("Results saved:", saved_zip)
print("ZIP SHA-256:", sha256(result_zip))
files.download(str(result_zip))